## About this Entry

Knowing how to leverage window functions is an extremely important skill that every data analyst should **definitely** have. In fact, they become omnipresent when we deal with time series analysis — a common need —, so it is crucial that we take our time to learn how to use them.

In contrast to the other SQL functionalities presented thus far, window functions do not calculate entities' features independently, neither do they aggregate values of all entities in a group. Instead, they perform aggregations within a moving window that can be determined by the user. But why is this important?

Suppose you work for a company whose sales data are stored, for analytical ends, at a daily level of granularity. Obviously, since it is unrealistic to expect that consumers will purchase goods at an uniform rate, there will be fluctuations in daily revenue. If you are trying to find out when during the month costumers are more likely buy your product — which would allow for a targeted marketing approach —, it is important to minimize the effect of arbitrary fluctuations in your analysis. This is when window functions become useful, since you can calculate, for each day, the average spending in the last few days. Therefore, instead of flagging days with high sales amounts as promising, we can make a more accurate recommendation based on periods, minimizing the effect of arbitrariness.

Let's see some concrete examples of window functions in action!

## Situation #1

The following code's goal is to identify the five highest grossing products sold in a store by category. To do this, it implements a window function that partitions the table by category and ranks products inside each partition by total sales amount. Equipped with each product's rank inside its category, I could filter the results to show only the top 5 highest selling products from each.

The tables are structured as follows:
### `orders`
| Column | Definition | Data type |
|--------|------------|-----------|
| `row_id`| Unique Record ID | `INTEGER` |
| `order_id` | Identifier for each order in table | `TEXT` |
| `order_date` | Date when order was placed | `TEXT` |
| `market` | Market order_id belongs to | `TEXT` |
| `region` | Region Customer belongs to | `TEXT` |
| `product_id` | Identifier of Product bought | `TEXT` |
| `sales` | Total Sales Amount for the Line Item | `DOUBLE PRECISION` |
| `quantity` | Total Quantity for the Line Item | `DOUBLE PRECISION` |
| `discount` | Discount applied for the Line Item | `DOUBLE PRECISION` |
| `profit` | Total Profit earned on the Line Item | `DOUBLE PRECISION` |

### `products`
| Column | Definition | Data type |
|--------|------------|-----------|
| `product_id`| Unique Identifier for the Product | `TEXT` |
| `category` | Category Product belongs to | `TEXT` |
| `sub_category` | Sub Category Product belongs to | `TEXT` |
| `product_name` | Detailed Name of the Product | `TEXT` |

A snapshot of each table is presented below, followed by the code and its results.

![Orders](<images/4th Entry/orders.png>)

![Products](<images/4th Entry/products.png>)

In [ ]:
WITH top_products_by_category AS (
	SELECT p.category,
		   p.product_name,
		   SUM(o.sales) AS product_total_sales,
		   SUM(o.profit) AS product_total_profit,
		   RANK() OVER(PARTITION BY p.category ORDER BY SUM(o.sales) DESC) AS product_rank
	  FROM orders AS o
	 	   INNER JOIN products AS p
		   USING(product_id)
	 GROUP BY category, product_name
	 ORDER BY category ASC, product_total_sales DESC
)
	
SELECT * FROM top_products_by_category WHERE product_rank <= 5;

![Top 5 Products.png](<images/4th Entry/top5_products.png>)

## Situation 2

The second situation we'll explore involves monitoring a manufacturing process using **statistical process control (SPC)** in order to identify and rectify problems early on. For this, the following code checks the uniformity of the produced parts using their height and raises an alert if it falls above or below certain predetermined thresholds. The formula provided for calculating upper and lower limits was the following:

$ucl = avg\_height + 3 * \frac{stddev\_height}{\sqrt{5}}$

$lcl = avg\_height - 3 * \frac{stddev\_height}{\sqrt{5}}$

Both the average height and the standard deviation are not calculated using all parts produced by each machine, but as the process goes on — otherwise we would only be able to identify defective parts at the end of the day. Therefore, we need a window function to calculate moving statistics that take into consideration the last N parts produced.

The table we will use is called "manufacturing parts" and have the following columns:
- `item_no`: the item number
- `length`: the length of the item made
- `width`: the width of the item made
- `height`: the height of the item made
- `operator`: the operating machine

A snapshot of it is presented below, followed by the code and the results produced by it.

![Manufacturing Process.png](<images/4th Entry/manufacturing_process.png>)

In [ ]:
WITH moving_stats AS (
	SELECT item_no,
		   ROW_NUMBER() OVER(PARTITION BY operator ORDER BY item_no ASC) AS row_number,
		   AVG(height) OVER(PARTITION BY operator ORDER BY item_no
							ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS avg_height,
		   STDDEV(height) OVER(PARTITION BY operator ORDER BY item_no
							   ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS stddev_height
	  FROM manufacturing_parts
)
	
SELECT p.operator,
	   s.row_number,
	   p.height,
	   s.avg_height,
	   s.stddev_height,
	   s.avg_height + 3*s.stddev_height/SQRT(5) AS ucl,
	   s.avg_height - 3*s.stddev_height/SQRT(5) AS lcl,
	   CASE
		 WHEN p.height NOT BETWEEN s.avg_height - 3*s.stddev_height/SQRT(5)
								   AND s.avg_height + 3*s.stddev_height/SQRT(5) THEN TRUE
		 ELSE FALSE
	   END AS alert
  FROM manufacturing_parts AS p
	   INNER JOIN moving_stats AS s
	   USING (item_no)
 WHERE row_number > 4
 ORDER BY item_no ASC;

![Alerts.png](<images/4th Entry/alerts.png>)

The codes presented above and their usefulness in everyday business scenarios clearly illustrate that window functions are highly valuable assets for data analysts and that knowing how to leverage them is indispensable. Also, in terms of data analysis, I believe that they are as complex as SQL gets since, ideally, we are using tables in read-only mode — data engineers will have to deal with a lot more. Therefore, as I believe I have already presented SQL's most important functionalities, the next entry in this portfolio will no longer be about it, but will instead introduced another crucial technology for data analysts: Python.